In [1]:
from alerce.core import Alerce
from astropy.table import vstack, Table
import matplotlib.pyplot as plt
import pyvo as vo
import requests
import sqlalchemy as sa
import sys
import numpy as np
from astropy.coordinates import SkyCoord
from astropy import units as u
import re
import george
import scipy.optimize as op
import emcee
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
alerce = Alerce()
alerce_tap = vo.dal.TAPService('https://tap.alerce.online/tap')

In [3]:
filename = "../Data/tns_Ia.csv"
sources = Table.read(filename, format="csv")
sources[0]

objid,name_prefix,name,ra,declination,redshift,typeid,type,reporting_groupid,reporting_group,source_groupid,source_group,discoverydate,discoverymag,discmagfilter,filter,reporters,time_received,internal_names,Discovery_ADS_bibcode,Class_ADS_bibcodes,creationdate,lastmodified
int64,str2,str8,float64,float64,float64,int64,str5,int64,str17,int64,str5,str23,float64,int64,str1,str454,str19,str146,str19,str61,str19,str19
207908,SN,2026lns,202.4570465,-5.7516547,0.0686,3,SN Ia,48,ZTF,48,ZTF,2026-05-07 05:03:27.996,19.2662,111,r,"Jesper Sollerman (SU), Christoffer Fremling (Caltech), Daniel Perley (LJMU), Theophile du Laz (Caltech) on behalf of ZTF",2026-05-07 09:15:10,"ZTF26aauvdne, GOTO26ejc, , ATLAS26fhx",2026TNSTR1975....1S,--,2026-05-07 09:15:12,2026-05-11 05:50:45


In [4]:
ZTF_mask = sources["source_group"] == "ZTF"
ZTF = sources[ZTF_mask]
ZTF[0]

objid,name_prefix,name,ra,declination,redshift,typeid,type,reporting_groupid,reporting_group,source_groupid,source_group,discoverydate,discoverymag,discmagfilter,filter,reporters,time_received,internal_names,Discovery_ADS_bibcode,Class_ADS_bibcodes,creationdate,lastmodified
int64,str2,str8,float64,float64,float64,int64,str5,int64,str17,int64,str5,str23,float64,int64,str1,str454,str19,str146,str19,str61,str19,str19
207908,SN,2026lns,202.4570465,-5.7516547,0.0686,3,SN Ia,48,ZTF,48,ZTF,2026-05-07 05:03:27.996,19.2662,111,r,"Jesper Sollerman (SU), Christoffer Fremling (Caltech), Daniel Perley (LJMU), Theophile du Laz (Caltech) on behalf of ZTF",2026-05-07 09:15:10,"ZTF26aauvdne, GOTO26ejc, , ATLAS26fhx",2026TNSTR1975....1S,--,2026-05-07 09:15:12,2026-05-11 05:50:45


In [5]:
ZTF_obj_Ia = []
pattern = r"ZTF"
for obj in ZTF["internal_names"]:
    #print(obj)
    if isinstance(obj, str):
        internal = obj.split(',')
        survay_name = [name for name in internal if re.search(pattern, name)]
        if survay_name:
            ZTF_obj_Ia.append(survay_name[0])

ZTF_obj_Ia = np.array(ZTF_obj_Ia)
print(len(ZTF_obj_Ia))

6425


In [6]:
def get_photometry(objectID):
    return alerce.query_forced_photometry(objectID, format="json")

In [18]:
max_curves = 20
light_curves = get_photometry(ZTF_obj_Ia[:20][..., None])
print(light_curves)

[]


In [14]:
with PdfPages('./SN_Ia_plots.pdf') as pdf:
    dic_curves = []
    for light in light_curves:
    
        if len(light) > 10:

            time = []
            mag = []
            error = []
            #initialize data
            for obs in light:
                if obs["fid"] == 2:
                    if obs["mag"] < 22:
                        time.append(obs["mjd"])
                        mag.append(obs["mag"])
                        error.append(obs["e_mag"])

            print(mag)
            mag = -1 * np.array(mag)
            mag_flipped = np.exp(mag)
            print(mag_flipped)
            error = np.array(error)
            error = np.exp(-1 * error)

            if len(mag_flipped) > 0:
                    
                fig, ax1 = plt.subplots()
                
                #plot raw lightcurve
                ax1.scatter(time, mag_flipped, marker='o')
                #ax1.invert_yaxis()
            
                #initialize data range
                x_fit = np.linspace(np.min(time) - 5, np.max(time) + 100, 1000)
            
                #get george off
                kernel =  np.var(mag_flipped) * george.kernels.ExpSquaredKernel(100)
                #kernel = np.var(mag_flipped) * george.kernels.Matern32Kernel(metric=750)
                gp = george.GP(kernel, solver=george.HODLRSolver)
                
                #use george
                gp.compute(time, error)
                #print(mag_flipped.shape, x_fit.shape)
                pred, pred_var = gp.predict(mag_flipped, x_fit, return_var=True)
    
                #find rise time
                peak = np.max(pred)
                print(peak)
                peak_index = np.argmax(pred)
                print(peak_index)
                slopes = np.gradient(pred, x_fit)
                rise_days = 0
                for point in x_fit[:peak_index]:
                    if slopes[np.where(x_fit == point)] < 1:
                        rise_days = point
                rise_time = rise_days - x_fit[0]
    
                #find down time
                down_days = 0
                for point in x_fit[peak_index:]:
                    if slopes[np.where(x_fit == point)] < 1:
                        down_days = point
                fall_time = down_days - x_fit[peak_index]
    
                #add curve to dictionary and append to list
                #print(table_curves)
                error = np.pow(pred_var, 0.5)
                dic = {"object": light[0]["oid"], "type": "SNIa", "mjd": x_fit, "mag": pred, "error": error, "peak": peak, "rise": rise_time, "fall": fall_time}
                dic_curves.append(dic)
                #print(table_curves)
    
                ax2 = ax1.twinx()
    
                #plot george
                ax2.plot(x_fit, pred, color='r')
            
                #ax1.fill_between(x_fit, pred - error, pred + error, interpolate=True)
                plt.title(f"Object: {light[0]["oid"]}; Type: SN_Ia; Peak: {peak}; Rise: {rise_time}; Fall: {fall_time}")
    
                #pdf.savefig()
                plt.show()
                plt.close()

table_curves = vstack(dic_curves)
table_curves.write('SNIa_test.ecsv', format='ascii.ecsv', overwrite=True)

ValueError: no values provided to stack.